In [1]:
%cd ..

c:\Users\namtv40\Projects\prefecthq-external-ingestion\ingestions


In [2]:
from dotenv import load_dotenv

load_dotenv()


True

In [3]:
import sys
import os

sys.path.insert(0, r"C:\Users\namtv40\Projects\prefecthq-external-ingestion\ingestions")

In [4]:
import os
import sys
import json
import time
import requests
import pandas as pd
import pyarrow as pa
from dateutil import parser
import pyarrow.parquet as pq
from datetime import datetime


In [5]:

if not os.path.exists("./tmp/data"):
    os.makedirs("./tmp/data")


In [6]:
from common.config import *
from common.http_util import *
from common.crawler_util import *
from common.ambari_util import *


def fetch_resource_name_freshwork(resource_name, records, **kwargs):
    print("Start crawl : ", resource_name)
    start_time = time.time()

    HDFS_BASE = "s3a://vcs-raw/finance-raw"
    # STATE_PATH = rc["state_path"]
    # BASE_URL = None
    # RESOURCE_URL = None
    # API_KEY_PATH = rc["api_key_path"]
    # API_COOKIE_PATH = rc["api_cookie_path"]
    # QUERY_PARAMS = None
    ENABLE_STATE =False
    HIVE_DB = "finance_raw"
    # crawl_mode = "modified_and_new"
    crawl_mode = kwargs.get("crawl_mode", "static")
    schema_local_path = None
    # result_json_key = "deleted_deals"

    print(resource_name)
    start_time = time.time()
    # =========================
    # MAIN
    # =========================
    if ENABLE_STATE:
        last_state = read_last_state(resource_name)
        print("Last state =", last_state)
    else:
        last_state = None

    if not records:
        print("No new data")
        out_of_data = True
        return True

    # =========================
    # Pandas → Parquet
    # =========================

    now = datetime.now()
    partition_path = "{}/{}".format(HDFS_BASE, resource_name)

    filename = "data_{}_{}{:02d}{:02d}_{}{:02d}{:02d}.parquet".format(
        resource_name, now.year, now.month, now.day, now.hour, now.minute, now.second
    )
    local_parquet = "./tmp/data/finance_raw/{}/{}".format(resource_name, filename)
    os.makedirs(os.path.dirname(local_parquet), exist_ok=True)

    # df.to_parquet(local_parquet,engine="pyarrow", compression="snappy", index=False)
    records = convert_json_add_ts_columns(records)

    schema_tm_path = "./resources/parquet_schema/finance_raw/{}.json".format(resource_name)
    schema = None
    if schema_local_path:
        schema = load_pyarrow_schema_from_json(schema_local_path)

    if os.path.exists(schema_tm_path):
        schema = load_pyarrow_schema_from_json(schema_tm_path)

    if not schema:
        schema = infer_schema_from_json(records)
        schema_json = save_pyarrow_type_to_json(schema)
        write_file_json(schema_tm_path, schema_json)

    records = convert_json_list_by_arrow_schema(records, schema)

    df = pd.DataFrame(records)
    data_table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)

    pq.write_table(data_table, local_parquet, compression="snappy")

    if crawl_mode == "static":
        replace_hdfs_https("{}".format(partition_path), local_parquet)
    else:
        upload_hdfs_https("{}".format(partition_path), local_parquet)

    print("Uploaded parquet to", partition_path)

    # =========================
    # Generate SQL (TEXT ONLY)
    # =========================
    sql = gen_spark_create_table(
        schema=schema,
        db=HIVE_DB,
        table=resource_name,
        location="{}/{}".format(HDFS_BASE, resource_name),
    )

    # filename = "create_table_{}_{}{:02d}{:02d}.sql".format(
    #     resource_name,
    #     now.hour,
    #     now.minute,
    #     now.second
    # )

    filename = "create_table_{}.sql".format(resource_name)

    local_sql = "./tmp/data/finance_raw/{}/{}".format(resource_name, filename)
    os.makedirs(os.path.dirname(local_sql), exist_ok=True)

    with open(local_sql, "w") as f:
        f.write(sql)

    # upload_hdfs_https(
    #     "{}/{}".format(HDFS_BASE, resource_name),
    #     local_sql
    # )

    print("Uploaded SQL definition")

    return False


In [7]:
import pandas as pd
import re
from pathlib import Path
from collections import defaultdict

import re
from collections import defaultdict
def excel_col_name(idx: int) -> str:
    """Zero-based index → Excel column (A, B, ..., AA)"""
    name = ""
    while idx >= 0:
        idx, rem = divmod(idx, 26)
        name = chr(rem + ord("A")) + name
        idx -= 1
    return name


def snake_case(text: str) -> str:
    text = (
        str(text)
        .strip()
        .lower()
        .replace("\n", " ")
    )
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text

def read_csv_and_normalize_columns(
    file_path: str,
    mapping: dict,
    header_row=0,
    drop_rows=0
) -> pd.DataFrame:
    # đọc raw, chưa set header
    df = pd.read_csv(file_path, header=None, encoding="utf-8", dtype=str)
    raw_headers = df.iloc[header_row]
    seen = defaultdict(int)
    new_columns = []

    for idx, col in enumerate(raw_headers):
        if pd.isna(col) or col == "":
            base = "nan"
        else:
            # 1️⃣ ưu tiên dictionary
            base = mapping.get(str(col).strip())

            # 2️⃣ fallback snake_case
            if base is None:
                base = snake_case(col)
                print(col)

        seen[base] += 1

        if seen[base] > 1:
            excel_col = excel_col_name(idx).lower()
            base = f"{base}__{excel_col}"

        new_columns.append(base)

    # gán columns sạch
    df.columns = new_columns

    # drop header rows
    df = df.iloc[drop_rows:].reset_index(drop=True)
    df = df.fillna("")

    return df

def read_folder_and_union_csv(
    folder_path: str,
    mapping: dict,
    header_row=0,
    drop_rows=0,
    encoding="utf-8"
):
    dfs = []
    folder = Path(folder_path)

    files = sorted(folder.glob("*.csv"))
    if not files:
        raise ValueError("❌ Không tìm thấy file CSV nào trong folder")

    for file in files:
        df = read_csv_and_normalize_columns(
            file_path=file,
            mapping=mapping,
            drop_rows=drop_rows,
            header_row=header_row
        )

        # trace file nguồn
        df["source_file"] = file.name
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

In [8]:
import pandas as pd
import re
from collections import defaultdict
def excel_col_name(idx: int) -> str:
    """Zero-based index → Excel column (A, B, ..., AA)"""
    name = ""
    while idx >= 0:
        idx, rem = divmod(idx, 26)
        name = chr(rem + ord("A")) + name
        idx -= 1
    return name


def snake_case(text: str) -> str:
    text = (
        str(text)
        .strip()
        .lower()
        .replace("\n", " ")
    )
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text

def read_excel_and_normalize_columns(
    file_path: str,
    mapping: dict,
    sheet_name=0,
    header_row=0,
    drop_rows=0
) -> pd.DataFrame:
    # đọc raw, chưa set header
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=None, dtype=str)

    raw_headers = df.iloc[header_row]
    seen = defaultdict(int)
    new_columns = []

    for idx, col in enumerate(raw_headers):
        if pd.isna(col) or col == "":
            base = "nan"
        else:
            # 1️⃣ ưu tiên dictionary
            base = mapping.get(col)

            # 2️⃣ fallback snake_case
            if base is None:
                print(f"Not found for col = '{col}'")
                base = snake_case(col)

        seen[base] += 1

        if seen[base] > 1:
            excel_col = excel_col_name(idx).lower()
            base = f"{base}__{excel_col}"

        new_columns.append(base)

    # gán columns sạch
    df.columns = new_columns

    # drop header rows
    df = df.iloc[drop_rows:].reset_index(drop=True)

    return df



In [ ]:
import re
import pandas as pd

def normalize_cell_text(value):
    if pd.isna(value):
        return value

    value = str(value).strip()                 # bỏ khoảng trắng đầu/cuối
    value = re.sub(r"\s+", " ", value)         # nhiều space -> 1 space
    value = value.title()                      # Bộ quốc phòng -> Bộ Quốc Phòng

    return value


In [ ]:

COLUMN_DICT_SPDV = {
    "Ngày": "date_key",
    "SPDV": "product_service_name",
  "Mã SPDV": "product_service_code",
  "Mã KM phí": "expense_code",
  "Khoản mục SPDV": "product_service_expense_category",
  "Data type": "data_type",
  "Thị trường": "market",
  "Triệu đồng": "amount_million_vnd",
  'THỊ TRƯỜNG': "market",
}

filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\fin-data-vudt\spdv.xlsx"
resource_name = "production_cost_allocation"

df = read_excel_and_normalize_columns(
      filename,
    sheet_name="Chi phí spdv",
    mapping=COLUMN_DICT_SPDV,
    header_row=0,
    drop_rows=1,
)
df["source_file"] = r'finance-data\fin-data-vudt\spdv.xlsx'


In [ ]:
df.head(5)

In [ ]:
# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open("tmp/" + resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")

fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

In [ ]:

REVENUE_MAPPING = {
    "Ngày": "invoice_date",
    # ===== Time =====
    "Ngày xuất hóa đơn": "invoice_date",
    "Tháng": "invoice_month",

    # ===== Invoice info =====
    "Xuất hóa đơn (HD)/ tạm tính (TT)": "invoice_type",  # HD / TT
    "Nội dung đề nghị TT(Theo chứng từ gốc)": "billing_description",
    "Nội dung đề nghị TT\n(Theo chứng từ gốc)": "billing_description",

    # ===== Revenue =====
    "Doanh thu": "revenue_amount",
    "Chia sẻ": "revenue_share_amount",
    "chia sẻ": "revenue_share_amount",
    "Doanh thu cuối": "final_revenue_amount",
    "Triệu đồng":"final_revenue_amount",

    "VAT": "vat_amount",
    "Tiền hàng (đã bao gồm VAT)": "gross_amount",
    "Tiền hàng \n(đã bao gồm VAT)": "gross_amount",
    "Tiền hàng": "gross_amount",

    # ===== Revenue recognition =====
    "DT dòng tiền đều/ DT lên 1 lần": "revenue_recognition_type",

    # ===== Product / Service =====
    "Phân loại SP/DV": "service_category",
    "SPDV cụ thể": "service_name",
    "SPDV":"service_name",
    "Mã SPDV": "service_code",

    # ===== Customer / Market =====
    "Khoản mục SPDV":"customer_scope",
    "Nội bộ/ Ngoài/QTế/Thị Trường": "customer_scope",  # internal / external / international / market
    "Phân loại KH": "customer_type",
    "Kênh khách hàng": "customer_channel",
    "Kênh khách hàng ": "customer_channel",
    "Khách hàng": "customer_name",
    "Segment": "customer_segment",
    "Nhóm khách hàng": "customer_group",
    

    # ===== Org =====
    "Phòng": "department",
    "AM": "account_manager",
    "AM hiện tại": "current_account_manager",
    "AM hiện tại ": "current_account_manager",
    "Presale": "presale",

    # ===== Classification =====
    "Phân loại DT Cũ/Mới": "revenue_type",  # new / existing
    "Phân loại SOC (SOC và non-SOC)": "soc_type",

    # ===== Revenue sharing =====
    "DT chia sẻ từ MSS": "mss_shared_revenue",

    # ===== Region =====
    "Bắc/Nam/Thị trường": "region_group",
    "Thị trường ":"region",
    "Nam/ Bắc": "region",   # ⚠️ duplicate semantic
    
    # ===== Flags =====
    "Vvip": "is_vvip_customer",
    "HĐ khung": "is_master_contract",
    "HĐ khung ":"is_master_contract",

    # ===== Notes =====
    "Ghi chú": "note",
}


filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\fin-data-vudt\spdv.xlsx"

df = read_excel_and_normalize_columns(
      filename,
    sheet_name="Doanh thu spdv",
    mapping=REVENUE_MAPPING,
    header_row=0,
    drop_rows=1,
)
df["source_file"] = r'finance-data\fin-data-vudt\spdv.xlsx'
df['vat_amount'] = df['vat_amount'].fillna('0')
df['revenue_share_amount'] = df['revenue_share_amount'].fillna('0')
df['gross_amount'] = df['gross_amount'].fillna('0')
df['revenue_amount'] = df['revenue_amount'].fillna('0')
df['final_revenue_amount'] = df['final_revenue_amount'].fillna('0')
df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce")
df["invoice_month"] = df["invoice_month"].fillna( df["invoice_date"].dt.strftime("%m"))

exclude_cols = ["gross_amount", "source_file", "vat_amount","revenue_share_amount" , "revenue_amount", "invoice_month", "final_revenue_amount"]
cols = df.columns.difference(exclude_cols)
df[cols] = df[cols].fillna("")

df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")

df["customer_scope"] = df["customer_scope"].apply(normalize_cell_text)

resource_name = "sales_revenue"
print(len(df))


In [ ]:
df.head(5)

In [ ]:
resource_name = "sales_revenue"
# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open("tmp/" + resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")
fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

In [ ]:

REVENUE_MAPPING = {
    "Ngày": "invoice_date",
    # ===== Time =====
    "Ngày xuất hóa đơn": "invoice_date",
    "Tháng": "invoice_month",

    # ===== Invoice info =====
    "Xuất hóa đơn (HD)/ tạm tính (TT)": "invoice_type",  # HD / TT
    "Nội dung đề nghị TT(Theo chứng từ gốc)": "billing_description",
    "Nội dung đề nghị TT\n(Theo chứng từ gốc)": "billing_description",

    # ===== Revenue =====
    "Doanh thu": "revenue_amount",
    "Chia sẻ": "revenue_share_amount",
    "chia sẻ": "revenue_share_amount",
    "Doanh thu cuối": "final_revenue_amount",
    "Triệu đồng":"final_revenue_amount",

    "VAT": "vat_amount",
    "Tiền hàng (đã bao gồm VAT)": "gross_amount",
    "Tiền hàng \n(đã bao gồm VAT)": "gross_amount",
    "Tiền hàng": "gross_amount",

    # ===== Revenue recognition =====
    "DT dòng tiền đều/ DT lên 1 lần": "revenue_recognition_type",

    # ===== Product / Service =====
    "Phân loại SP/DV": "service_category",
    "SPDV cụ thể": "service_name",
    "SPDV":"service_name",
    "Mã SPDV": "service_code",

    # ===== Customer / Market =====
    "Khoản mục SPDV":"customer_scope",
    "Nội bộ/ ngoài": "customer_scope",
    "Nội bộ/ Ngoài/QTế/Thị Trường": "customer_scope",  # internal / external / international / market
    "Phân loại KH": "customer_type",
    "Kênh khách hàng": "customer_channel",
    "Kênh khách hàng ": "customer_channel",
    "Khách hàng": "customer_name",
    "Segment": "customer_segment",
    "Nhóm khách hàng": "customer_group",
    
    # "Phân loại AM (Nội bộ/VVIP/Unname": "am_segment",

    # ===== Org =====
    "Phòng": "department",
    "AM": "account_manager",
    "AM hiện tại": "current_account_manager",
    "AM hiện tại ": "current_account_manager",
    "AM hiện tại .1": "current_account_manager",
    "Presale": "presale",

    # ===== Classification =====
    "Phân loại DT Cũ/Mới": "revenue_type",  # new / existing
    "Phân loại SOC (SOC và non-SOC)": "soc_type",

    # ===== Revenue sharing =====
    "DT chia sẻ từ MSS": "mss_shared_revenue",

    # ===== Region =====
    "Bắc/Nam/Thị trường": "region_group",
    "Thị trường ":"region",
    "Thị trường/Bắc Nam-SPDV":"region",
    "Nam/ Bắc": "region",   # ⚠️ duplicate semantic
    
    "TT NAM TT BẮC":"region_group",
    
    # ===== Flags =====
    "Vvip": "is_vvip_customer",
    "HĐ khung": "is_master_contract",
    "HĐ khung ":"is_master_contract",

    # ===== Notes =====
    "Ghi chú": "note",
}


filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\KD\TĐ_THANG 6_step_2_MSS_share_TI.xlsx"

df = read_excel_and_normalize_columns(
      filename,
    sheet_name="Sheet1",
    mapping=REVENUE_MAPPING,
    header_row=0,
    drop_rows=1,
)
df["source_file"] = r'finance-data\KD\TĐ_THANG 6_step_2_MSS_share_TI.xlsx'
df['vat_amount'] = df['vat_amount'].fillna('0')
df['revenue_share_amount'] = df['revenue_share_amount'].fillna('0')
df['gross_amount'] = df['gross_amount'].fillna('0')
df['revenue_amount'] = df['revenue_amount'].fillna('0')
df['final_revenue_amount'] = df['final_revenue_amount'].fillna('0')
df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce")
df["invoice_month"] = df["invoice_month"].fillna( df["invoice_date"].dt.strftime("%m"))

exclude_cols = ["gross_amount", "source_file", "vat_amount","revenue_share_amount" , "revenue_amount", "invoice_month", "final_revenue_amount"]
cols = df.columns.difference(exclude_cols)
df[cols] = df[cols].fillna("")

df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")

df["customer_scope"] = df["customer_scope"].apply(normalize_cell_text)

resource_name = "sales_revenue"
print(len(df))


Not found for col = 'TransactionId'
Not found for col = 'Phân loại AM (Nội bộ/VVIP/Unname'
Not found for col = 'VIP/ VVIP theo TTr'
Not found for col = 'Note'
870


C:\Users\namtv40\AppData\Local\Temp\ipykernel_8832\1834754884.py:94: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce")


In [23]:
df.head(5)

,transactionid,invoice_date,invoice_month,invoice_type,billing_description,revenue_amount,revenue_share_amount,final_revenue_amount,vat_amount,gross_amount,...,soc_type,mss_shared_revenue,region,service_code,region_group,current_account_manager__ac,phân_loại_am_nội_bộvvipunname,vip_vvip_theo_ttr,note,source_file
0,1,2026-06-30 00:00:00,T6,HD,Kênh truyền MetroWAN 20Mbps từ 09/01/2026 đến ...,21027916,0,21027916,0,21027916,...,non-SOC,No,TT Miền bắc,VCS014,TT Miền bắc,Trangpn1,Nội bộ,,,finance-data\KD\TĐ_THANG 6_step_2_MSS_share_TI...
1,3,2026-06-30 00:00:00,T6,HD,Dịch vụ bảo vệ website cloudrity từ 09/01/2026...,8986298,0,8986298,0,8986298,...,non-SOC,No,TT Miền bắc,VCS013,TT Miền bắc,Trangpn1,Nội bộ,,,finance-data\KD\TĐ_THANG 6_step_2_MSS_share_TI...
2,6,2026-06-30 00:00:00,T6,HD,"Phần mềm giám sát an ninh mạng, thu thập dữ li...",780000000,0,780000000,0,780000000,...,non-SOC,No,TT Miền bắc,VCS002,TT Miền bắc,Trangpn1,Nội bộ,,,finance-data\KD\TĐ_THANG 6_step_2_MSS_share_TI...
3,7,2026-06-30 00:00:00,T6,HD,"Phần mềm điều phối, tự động hoá và phản ứng an...",2080000000,0,2080000000,0,2080000000,...,non-SOC,No,TT Miền bắc,VCS003,TT Miền bắc,Trangpn1,Nội bộ,,,finance-data\KD\TĐ_THANG 6_step_2_MSS_share_TI...
4,8,2026-06-30 00:00:00,T6,HD,Phần mềm phát hiện và phản ứng sự cố ATTT (Vie...,316680000,0,316680000,0,316680000,...,non-SOC,No,TT Miền bắc,VCS004,TT Miền bắc,Trangpn1,Nội bộ,,,finance-data\KD\TĐ_THANG 6_step_2_MSS_share_TI...


In [24]:
resource_name = "sales_revenue"
# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open("tmp/" + resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")
fetch_resource_name_freshwork(resource_name, records, crawl_mode="modified_and_new")

✅ Done: sales_revenue.json created
Start crawl :  sales_revenue
sales_revenue
Add Upload  s3a://vcs-raw/finance-raw/sales_revenue ./tmp/data/finance_raw/sales_revenue/data_sales_revenue_20260713_90226.parquet
bucket=vcs-raw , key=finance-raw/sales_revenue/data_sales_revenue_20260713_90226.parquet
Uploaded parquet to s3a://vcs-raw/finance-raw/sales_revenue
Uploaded SQL definition


False